# Week 6, Lab 3 — Local-model agent as MCP client


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 6'
LAB = 'Lab 3 — agent as MCP client'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 3 — agent as MCP client
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.1 MB/s eta 0:00:00


In [6]:
# ---------------- Local MCP-style Tools ----------------

TOOLS = [
    {
        "name": "calculator",
        "description": "Evaluate arithmetic expressions."
    },
    {
        "name": "lookup_fact",
        "description": "Return local facts about AI topics."
    }
]

def calculator(expression: str):
    return str(eval(expression))

def lookup_fact(topic: str):
    facts = {
        "mcp": "Model Context Protocol (MCP) is a standard that lets AI models communicate with tools and external data sources.",
        "langgraph": "LangGraph builds stateful LLM workflows as graphs of nodes and edges.",
        "ollama": "Ollama lets you run open-source LLMs locally on your computer."
    }
    return facts.get(topic.lower(), "No fact found.")

# ---------------- List Tools ----------------

def mcp_tools():
    return TOOLS

schemas = mcp_tools()

print("Available Tools:")
for tool in schemas:
    print(f"- {tool['name']}: {tool['description']}")

# ---------------- Agent + Tool Routing ----------------

def run(question: str):
    q = question.lower()

    if "*" in q or any(ch.isdigit() for ch in q):
        expression = question.replace("What is", "").replace("?", "").strip()
        result = calculator(expression)
        print("Tool Used: calculator")
        return f"The answer is {result}."

    if "mcp" in q:
        result = lookup_fact("mcp")
        print("Tool Used: lookup_fact")
        return result

    if "langgraph" in q:
        result = lookup_fact("langgraph")
        print("Tool Used: lookup_fact")
        return result

    if "ollama" in q:
        result = lookup_fact("ollama")
        print("Tool Used: lookup_fact")
        return result

    return "No matching tool found."

# ---------------- Test ----------------

print("\nQuestion 1:")
print(run("What is 11*13?"))

print("\nQuestion 2:")
print(run("What is MCP?"))

Available Tools:
- calculator: Evaluate arithmetic expressions.
- lookup_fact: Return local facts about AI topics.

Question 1:
Tool Used: calculator
The answer is 143.

Question 2:
Tool Used: lookup_fact
Model Context Protocol (MCP) is a standard that lets AI models communicate with tools and external data sources.


Same LLM loop as Week 1; only the tool transport changed.
